# CGR-MAT v1.1 — Fixed Multimodal Severity Pipeline

This hotfix uses the official `Severity` labels as the authoritative cohort definition, restoring the expected **N=463 / complicated=118** cohort. It also exports a transparent diagnosis–severity audit for the 18 records removed by the earlier incorrect exact-diagnosis filter.

**Run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Allow Google Drive access. Existing verified dataset files in Drive are reused automatically.


In [ ]:
!pip -q install catboost==1.2.8 openpyxl==3.1.5


In [ ]:
import os
import torch

os.environ["CGR_MAT_RUN_MODE"] = "full"
os.environ["CGR_MAT_USE_DRIVE"] = "1"
os.environ["CGR_MAT_FORCE_RESTART"] = "0"
os.environ["CGR_MAT_PRETRAINED"] = "1"

if os.environ["CGR_MAT_RUN_MODE"] == "full" and not torch.cuda.is_available():
    raise RuntimeError("Full mode requires a GPU runtime. Select Runtime → Change runtime type → T4 GPU.")
print("Run mode:", os.environ["CGR_MAT_RUN_MODE"])
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
import hashlib
import urllib.request

SOURCE_COMMIT = "d3b385004c689b2e9a098cdac0c520f948716d3b"
LOADER_URL = (
    "https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/"
    f"{SOURCE_COMMIT}/src/cgr_mat/cgr_mat_verified_loader_v1_1.py"
)
EXPECTED_LOADER_SHA256 = "fbcc6b57b6d31d763b3b5a2f2f1db655211546fd68d61965922771a9331ab5a5"

print("Loading pinned CGR-MAT v1.1 hotfix loader...")
print("Source commit:", SOURCE_COMMIT)
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
actual_loader_sha256 = hashlib.sha256(loader_bytes).hexdigest()
if actual_loader_sha256 != EXPECTED_LOADER_SHA256:
    raise RuntimeError(
        "Loader integrity failure.\n"
        f"Expected: {EXPECTED_LOADER_SHA256}\nActual:   {actual_loader_sha256}"
    )
print("✓ CGR-MAT v1.1 loader integrity verified.")
loader = loader_bytes.decode("utf-8")
exec(compile(loader, LOADER_URL, "exec"), globals(), globals())


## Expected cohort audit

Before training begins, the notebook must show `severity_labelled_cohort = 463`, `complicated = 118`, and `uncomplicated = 345`. It saves `severity_cohort_diagnosis_audit.csv` and `severity_cohort_nonexact_diagnosis_rows.csv` for transparent review.

All checkpoints, `.pt`, `.pkl`, `.cbm`, `.csv`, `.json`, `.png`, checksum manifest and ZIP bundle are saved under `MyDrive/MAT-Appendix/cgr_mat_runs/`.
